In [1]:
import mne
import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pywt
import warnings
warnings.filterwarnings('ignore')

data_path = r"C:\Users\hiro2\Documents\EEG_project\data\MUSIN-G"
save_path = r"C:\Users\hiro2\Documents\EEG_project\data\scalograms"
os.makedirs(save_path, exist_ok=True)

MUSE_CH = [50, 26, 88, 7]
MUSE_CH_NAMES = ['AF7', 'AF8', 'TP9', 'TP10']
SF = 250

# 対応する周波数帯（θ/α/β）をカバーするスケール
freqs_target = np.linspace(1, 40, 64)  # 1〜40Hz、64点
scales = pywt.frequency2scale('morl', freqs_target / SF)

for sub_id in range(1, 21):
    for ses_id in range(1, 13):
        set_path = os.path.join(
            data_path,
            f"sub-{sub_id:03d}",
            f"ses-{ses_id:02d}",
            "eeg",
            f"sub-{sub_id:03d}_ses-{ses_id:02d}_task-MusicListening_run-{ses_id}_eeg.set"
        )
        if not os.path.exists(set_path):
            continue

        try:
            raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
            raw.filter(1., 40., fir_window='hamming', verbose=False)
            data = raw.get_data()[MUSE_CH]  # (4ch, timepoints)

            scal_list = []
            for ch_data in data:
                # ダウンサンプリング（計算量削減）
                ch_down = ch_data[::4]  # 250Hz → 62.5Hz
                coef, _ = pywt.cwt(ch_down, scales, 'morl')
                power = np.abs(coef) ** 2
                power = np.log1p(power)
                scal_list.append(power)

            # (4ch, freq_bins, time_bins)
            scal = np.stack(scal_list, axis=0)

            fname = f"sub{sub_id:03d}_ses{ses_id:02d}.npy"
            np.save(os.path.join(save_path, fname), scal)

            print(f"✓ sub-{sub_id:03d} ses-{ses_id:02d} shape={scal.shape}")

        except Exception as e:
            print(f"✗ sub-{sub_id:03d} ses-{ses_id:02d}: {e}")

print(f"\n完了！{save_path} に保存")

# サンプル可視化
sample = np.load(os.path.join(save_path, "sub001_ses01.npy"))
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    ax.imshow(sample[i], aspect='auto', origin='lower',
              cmap='magma', extent=[0, sample.shape[2], 1, 40])
    ax.set_title(MUSE_CH_NAMES[i])
    ax.set_xlabel('Time')
    ax.set_ylabel('Frequency (Hz)')
plt.tight_layout()
plt.savefig(r"C:\Users\hiro2\Documents\EEG_project\data\sample_scalogram.png", dpi=150)
print("スカログラム画像保存完了")

✓ sub-001 ses-01 shape=(4, 64, 8503)
✓ sub-001 ses-02 shape=(4, 64, 7808)
✓ sub-001 ses-03 shape=(4, 64, 8924)
✓ sub-001 ses-04 shape=(4, 64, 7593)
✓ sub-001 ses-05 shape=(4, 64, 8413)
✓ sub-001 ses-06 shape=(4, 64, 6880)
✓ sub-001 ses-07 shape=(4, 64, 7907)
✓ sub-001 ses-08 shape=(4, 64, 8238)
✓ sub-001 ses-09 shape=(4, 64, 8532)
✓ sub-001 ses-10 shape=(4, 64, 8719)
✓ sub-001 ses-11 shape=(4, 64, 7697)
✓ sub-001 ses-12 shape=(4, 64, 7989)
✓ sub-002 ses-01 shape=(4, 64, 34013)
✓ sub-002 ses-02 shape=(4, 64, 31229)
✓ sub-002 ses-03 shape=(4, 64, 35696)
✓ sub-002 ses-04 shape=(4, 64, 30367)
✓ sub-002 ses-05 shape=(4, 64, 33646)
✓ sub-002 ses-06 shape=(4, 64, 27517)
✓ sub-002 ses-07 shape=(4, 64, 31630)
✓ sub-002 ses-08 shape=(4, 64, 32950)
✓ sub-002 ses-09 shape=(4, 64, 34129)
✓ sub-002 ses-10 shape=(4, 64, 34871)
✓ sub-002 ses-11 shape=(4, 64, 30788)
✓ sub-002 ses-12 shape=(4, 64, 31954)
✓ sub-003 ses-01 shape=(4, 64, 8504)
✓ sub-003 ses-02 shape=(4, 64, 7808)
✓ sub-003 ses-03 shape=(4,